# Dependencies and Definitions

In [1]:
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import logging

from model import Model, Transformer, TransformerBlock
from train import train
from infer import load_model, run_model
from dataset import TextDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_dir = Path("checkpoints") # Change me for inference!
model_file = "0.pt"

prompt = "= world war ii =\n" # The model really really likes this subject

# Inference

In [2]:
# INFERENCE, CHANGE MODEL PATH ABOVE BEFORE RUNNING

model = load_model(model_dir / model_file)
output = run_model(prompt, model, max_tokens=100)
print(output)

= world war ii =
 The Royal Navy ( 1777 ) . She began in a period of war , with the outbreak of the Second World War in 1776 . She was frequently used in the Battle of Britain , including a period of conflict , and was used as a target of the blockade of the French blockade . In 1730 , however , she was sent to the Mediterranean and then through the French blockade , a blockade in Spain , with the British fleet serving a fleet of the line fleet . She was stricken from the Navy


# Train

## Sanity Checks

In [ ]:
# TRANSFORMER BLOCK TEST

batch, context, dim = 16, 100, 512
heads = 4
test_attn = TransformerBlock(embed_dim=dim, num_heads=heads).to(device)

# Forward
x = torch.randn(batch, context, dim).to(device)
test_output = test_attn(x)
print("Self attention shape -", "✅" if test_output.shape == (batch, context, dim) else "❎")

In [ ]:
# TRANSFORMER TEST

batch, context, dim = 16, 100, 512
test_transformer = Transformer(embed_dim=dim).to(device)

# Forward
x = torch.randn(batch, context, dim).to(device)
test_output = test_transformer(x)
print("Transformer shape -", "✅" if test_output.shape == (batch, context, dim) else "❎")

In [ ]:
# MODEL TEST

batch, context, vocab = 4, 1024, 32768
test_model = Model(vocab_size=vocab, max_context=context).to(device)

# Forward
x = torch.randint(0, vocab, size=(batch, context)).to(device)
test_output = test_model(x)
print("Model shape -", "✅" if test_output.shape == (batch, context, vocab) else "❎")

# Parameters
total_params = sum(p.numel() for p in test_model.parameters())
print(f"Model parameters - {total_params / 1e6:.2f}M")

## Train

In [ ]:
logging.basicConfig(level=logging.INFO, filename="train.log", filemode="w")
train()